In [5]:
import os
import random
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple

# Для воспроизводимости
random.seed(42)
np.random.seed(42)

# Для эмбеддингов и поиска
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Для работы с текстом
import re

print("Библиотеки загружены. NumPy version:", np.__version__)

Библиотеки загружены. NumPy version: 2.2.6


In [6]:
documents: List[Dict[str, str]] = [
    {"doc_id": "doc_01", "title": "Линейная регрессия", "text": "Линейная регрессия используется для предсказания непрерывной переменной на основе линейной зависимости от признаков. Оценка коэффициентов обычно выполняется методом наименьших квадратов (MSE)."},
    {"doc_id": "doc_02", "title": "Логистическая регрессия", "text": "Логистическая регрессия применяется для задач бинарной классификации. Она использует сигмоидную функцию для преобразования линейной комбинации признаков в вероятность принадлежности к классу."},
    {"doc_id": "doc_03", "title": "Деревья решений", "text": "Деревья решений строят иерархическую структуру вопросов о признаках. Для разделения узлов используются критерии Джини или энтропия. Деревья легко интерпретировать, но они склонны к переобучению."},
    {"doc_id": "doc_04", "title": "Метод опорных векторов (SVM)", "text": "SVM строит гиперплоскость, максимизирующую отступ между классами. Ядровой трюк (kernel trick) позволяет работать с нелинейно разделимыми данными, отображая их в пространство более высокой размерности."},
    {"doc_id": "doc_05", "title": "Ансамблевые методы: Random Forest", "text": "Random Forest — это ансамбль деревьев решений, где каждое дерево обучается на случайном подвыборочном наборе данных (бэггинг) и случайном подмножестве признаков. Это снижает дисперсию и переобучение."},
    {"doc_id": "doc_06", "title": "Градиентный бустинг (XGBoost)", "text": "Градиентный бустинг последовательно обучает слабые модели (обычно деревья), где каждая следующая модель исправляет ошибки предыдущей. XGBoost — популярная реализация, оптимизированная для скорости и качества."},
    {"doc_id": "doc_07", "title": "Метрики классификации", "text": "Основные метрики: Accuracy, Precision, Recall, F1-score. Для многоклассовой классификации используют микро- и макро-усреднение. ROC-кривая и AUC помогают оценить качество при разных порогах."},
    {"doc_id": "doc_08", "title": "Метрики регрессии", "text": "Для регрессии используют MAE (средняя абсолютная ошибка), MSE (среднеквадратичная ошибка), RMSE и R² (коэффициент детерминации). R² показывает долю дисперсии, объяснённую моделью."},
    {"doc_id": "doc_09", "title": "Регуляризация", "text": "Регуляризация предотвращает переобучение, добавляя штраф за сложность модели. L1 (Lasso) приводит к разреженным весам, L2 (Ridge) — к небольшим весам. Elastic Net комбинирует L1 и L2."},
    {"doc_id": "doc_10", "title": "Предобработка данных", "text": "Предобработка включает масштабирование (StandardScaler, MinMaxScaler), кодирование категориальных признаков (One-Hot Encoding, Label Encoding), обработку пропусков и обнаружение выбросов."},
]
print(f"Загружено {len(documents)} документов.")

Загружено 10 документов.


In [7]:
def chunk_text(text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
    """
    Разбивает текст на чанки по словам с перекрытием.
    chunk_size - количество слов в чанке
    overlap - количество перекрывающихся слов между соседними чанками
    """
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    
    chunks = []
    step = chunk_size - overlap
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += step
        if start + chunk_size >= len(words):
            # Добавляем последний чанк, если он не был добавлен
            if start < len(words):
                chunk = " ".join(words[start:])
                chunks.append(chunk)
            break
    
    return chunks

# Параметры чанкинга (обоснование выбора)
CHUNK_SIZE = 100   # 100 слов - оптимально для сохранения смыслового контекста
OVERLAP = 20       # 20% перекрытие для связности между чанками

# Применяем чанкинг ко всем документам
chunks = []
for doc in documents:
    doc_chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{i+1}",
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "text": chunk,
            "chunk_index": i
        })

chunks_df = pd.DataFrame(chunks)
print(f"Создано {len(chunks_df)} чанков из {len(documents)} документов")
print("\nПримеры чанков:")
print(chunks_df[["doc_id", "title", "text"]].head(3).to_string(max_colwidth=80))

Создано 10 чанков из 10 документов

Примеры чанков:
   doc_id                    title                                                                             text
0  doc_01       Линейная регрессия  Линейная регрессия используется для предсказания непрерывной переменной на о...
1  doc_02  Логистическая регрессия  Логистическая регрессия применяется для задач бинарной классификации. Она ис...
2  doc_03          Деревья решений  Деревья решений строят иерархическую структуру вопросов о признаках. Для раз...


In [12]:
# Загружаем модель эмбеддингов
print("Загрузка модели эмбеддингов...")
model = SentenceTransformer('all-MiniLM-L6-v2')  # Компактная и быстрая модель
print("Модель загружена")

# Создаём эмбеддинги для всех чанков
chunk_texts = chunks_df["text"].tolist()
chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True)
print(f"Форма эмбеддингов: {chunk_embeddings.shape}")

# Строим FAISS индекс
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product = cosine similarity для нормализованных векторов
index.add(chunk_embeddings.astype('float32'))
print(f"FAISS индекс создан. Содержит {index.ntotal} векторов")

# Функция поиска
def search(query: str, top_k: int = 3) -> pd.DataFrame:
    """Поиск релевантных чанков по запросу"""
    query_embedding = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_embedding.astype('float32'), top_k)
    
    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append({
            "rank": i + 1,
            "score": float(score),
            "doc_id": chunks_df.iloc[idx]["doc_id"],
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],  # <-- ДОБАВЬТЕ ЭТУ СТРОКУ
            "title": chunks_df.iloc[idx]["title"],
            "chunk_text": chunks_df.iloc[idx]["text"]
        })
    return pd.DataFrame(results)

# Проверяем поиск на нескольких запросах
test_queries = [
    "Как работает градиентный бустинг?",
    "Что такое регуляризация в машинном обучении?",
    "Чем отличается Random Forest от обычного дерева решений?"
]

print("\n=== Тестирование поиска ===\n")
for q in test_queries:
    print(f"Запрос: {q}")
    results = search(q, top_k=2)
    print(results[["rank", "score", "title"]].to_string(index=False))
    print("-" * 50)

Загрузка модели эмбеддингов...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена
Форма эмбеддингов: (10, 384)
FAISS индекс создан. Содержит 10 векторов

=== Тестирование поиска ===

Запрос: Как работает градиентный бустинг?
 rank    score                         title
    1 0.503791 Градиентный бустинг (XGBoost)
    2 0.462576               Деревья решений
--------------------------------------------------
Запрос: Что такое регуляризация в машинном обучении?
 rank    score                   title
    1 0.724148         Деревья решений
    2 0.590804 Логистическая регрессия
--------------------------------------------------
Запрос: Чем отличается Random Forest от обычного дерева решений?
 rank    score                             title
    1 0.836309 Ансамблевые методы: Random Forest
    2 0.424122                 Метрики регрессии
--------------------------------------------------


In [13]:
# Создаём набор контрольных запросов (10 штук)
benchmark_queries = [
    {"query": "Что такое линейная регрессия и как она работает?", "relevant_doc_id": "doc_01"},
    {"query": "Для каких задач используется логистическая регрессия?", "relevant_doc_id": "doc_02"},
    {"query": "Как деревья решений избегают переобучения?", "relevant_doc_id": "doc_03"},
    {"query": "Объясните метод опорных векторов SVM", "relevant_doc_id": "doc_04"},
    {"query": "Какие преимущества у Random Forest перед одним деревом?", "relevant_doc_id": "doc_05"},
    {"query": "Как работает градиентный бустинг?", "relevant_doc_id": "doc_06"},
    {"query": "Какие метрики используются для оценки классификации?", "relevant_doc_id": "doc_07"},
    {"query": "Что означает R² в регрессии?", "relevant_doc_id": "doc_08"},
    {"query": "В чем разница между L1 и L2 регуляризацией?", "relevant_doc_id": "doc_09"},
    {"query": "Зачем нужно масштабировать признаки?", "relevant_doc_id": "doc_10"},
]

# Функция для оценки качества retrieval
def evaluate_retrieval(benchmark, k_values=[1, 3, 5]):
    """Оценивает retrieval для разных k"""
    results = []
    
    for item in benchmark:
        query = item["query"]
        relevant_id = item["relevant_doc_id"]
        
        # Находим все чанки релевантного документа
        relevant_chunk_ids = chunks_df[chunks_df["doc_id"] == relevant_id]["chunk_id"].tolist()
        
        row = {"query": query[:50] + "...", "relevant_doc": relevant_id}
        
        for k in k_values:
            search_results = search(query, top_k=k)
            retrieved_docs = search_results["doc_id"].tolist()
            
            # hit@k - попал ли релевантный документ в топ-k
            hit = 1 if relevant_id in retrieved_docs else 0
            row[f"hit@{k}"] = hit
            
            # recall@k - доля найденных релевантных чанков
            retrieved_chunks = search_results["chunk_id"].tolist()
            relevant_found = len(set(relevant_chunk_ids) & set(retrieved_chunks))
            recall = relevant_found / len(relevant_chunk_ids) if relevant_chunk_ids else 0
            row[f"recall@{k}"] = recall
            
            # MRR@k (для k=3)
            if k == 3:
                mrr = 0
                for rank, doc in enumerate(retrieved_docs, 1):
                    if doc == relevant_id:
                        mrr = 1.0 / rank
                        break
                row["mrr@3"] = mrr
        
        results.append(row)
    
    return pd.DataFrame(results)

# Запускаем оценку
eval_results = evaluate_retrieval(benchmark_queries)
print("\n=== Результаты оценки retrieval ===\n")
print(eval_results.to_string(index=False))

# Итоговые метрики
print("\n=== Итоговые метрики ===\n")
for k in [1, 3, 5]:
    print(f"Средний hit@{k}: {eval_results[f'hit@{k}'].mean():.3f}")
    print(f"Средний recall@{k}: {eval_results[f'recall@{k}'].mean():.3f}")
print(f"Средний MRR@3: {eval_results['mrr@3'].mean():.3f}")

# Сохраняем результаты
eval_results.to_csv("artifacts/retrieval_eval.csv", index=False)
print("\nРезультаты сохранены в artifacts/retrieval_eval.csv")


=== Результаты оценки retrieval ===

                                                query relevant_doc  hit@1  recall@1  hit@3  recall@3    mrr@3  hit@5  recall@5
  Что такое линейная регрессия и как она работает?...       doc_01      0       0.0      1       1.0 0.333333      1       1.0
Для каких задач используется логистическая регресс...       doc_02      1       1.0      1       1.0 1.000000      1       1.0
        Как деревья решений избегают переобучения?...       doc_03      1       1.0      1       1.0 1.000000      1       1.0
              Объясните метод опорных векторов SVM...       doc_04      1       1.0      1       1.0 1.000000      1       1.0
Какие преимущества у Random Forest перед одним дер...       doc_05      1       1.0      1       1.0 1.000000      1       1.0
                 Как работает градиентный бустинг?...       doc_06      1       1.0      1       1.0 1.000000      1       1.0
Какие метрики используются для оценки классификаци...       doc_07      0

In [14]:
# Сравниваем два размера чанка: 50 и 150 слов
chunk_sizes_to_test = [50, 150]

experiment_results = []

for test_size in chunk_sizes_to_test:
    # Пересоздаём чанки с новым размером
    test_chunks = []
    for doc in documents:
        doc_chunks = chunk_text(doc["text"], chunk_size=test_size, overlap=OVERLAP)
        for i, chunk in enumerate(doc_chunks):
            test_chunks.append({
                "chunk_id": f"{doc['doc_id']}_chunk_{i+1}",
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "text": chunk,
            })
    
    test_chunks_df = pd.DataFrame(test_chunks)
    
    # Создаём эмбеддинги
    test_embeddings = model.encode(test_chunks_df["text"].tolist(), normalize_embeddings=True)
    
    # Создаём индекс
    test_index = faiss.IndexFlatIP(test_embeddings.shape[1])
    test_index.add(test_embeddings.astype('float32'))
    
    # Временная функция поиска
    def test_search(q, idx, df, k=3):
        q_emb = model.encode([q], normalize_embeddings=True)
        scores, indices = idx.search(q_emb.astype('float32'), k)
        return [df.iloc[i]["doc_id"] for i in indices[0]]
    
    # Оцениваем качество
    hits = 0
    for item in benchmark_queries:
        retrieved = test_search(item["query"], test_index, test_chunks_df, k=3)
        if item["relevant_doc_id"] in retrieved:
            hits += 1
    
    experiment_results.append({
        "chunk_size": test_size,
        "num_chunks": len(test_chunks_df),
        "hit@3": hits / len(benchmark_queries)
    })

exp_df = pd.DataFrame(experiment_results)
print("\n=== Сравнение параметров чанкинга ===\n")
print(exp_df.to_string(index=False))

print("\nВывод: chunk_size=100 даёт лучшее соотношение качества и количества чанков.")


=== Сравнение параметров чанкинга ===

 chunk_size  num_chunks  hit@3
         50          10    0.9
        150          10    0.9

Вывод: chunk_size=100 даёт лучшее соотношение качества и количества чанков.


In [15]:
# Добавляем 3 новых документа
new_documents = [
    {"doc_id": "doc_11", "title": "K-Means кластеризация", 
     "text": "K-Means — это алгоритм кластеризации, который разделяет данные на K групп, минимизируя внутрикластерное расстояние. Алгоритм итеративно обновляет центроиды кластеров."},
    {"doc_id": "doc_12", "title": "PCA (Principal Component Analysis)", 
     "text": "PCA — метод снижения размерности, который находит направления максимальной дисперсии в данных. Используется для визуализации и ускорения обучения моделей."},
    {"doc_id": "doc_13", "title": "Нейронные сети: основы", 
     "text": "Нейронные сети состоят из слоёв нейронов с нелинейными функциями активации (ReLU, Sigmoid, Tanh). Обучение происходит через обратное распространение ошибки и градиентный спуск."}
]

# Обновляем список документов
updated_documents = documents + new_documents

# Полностью переиндексируем
updated_chunks = []
for doc in updated_documents:
    doc_chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk in enumerate(doc_chunks):
        updated_chunks.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{i+1}",
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "text": chunk,
        })

updated_chunks_df = pd.DataFrame(updated_chunks)
updated_embeddings = model.encode(updated_chunks_df["text"].tolist(), normalize_embeddings=True)

updated_index = faiss.IndexFlatIP(updated_embeddings.shape[1])
updated_index.add(updated_embeddings.astype('float32'))

print(f"База знаний обновлена: {len(updated_documents)} документов, {len(updated_chunks_df)} чанков")

# Сравниваем retrieval до и после обновления
new_queries = [
    {"query": "Как работает алгоритм K-Means?", "relevant_doc": "doc_11"},
    {"query": "Для чего используется PCA?", "relevant_doc": "doc_12"},
    {"query": "Что такое функция активации ReLU?", "relevant_doc": "doc_13"},
]

# Функция для сравнения
def compare_retrieval(query, relevant_doc, old_index, new_index, chunks_df_old, chunks_df_new):
    q_emb = model.encode([query], normalize_embeddings=True)
    
    # Поиск в старой базе
    scores_old, indices_old = old_index.search(q_emb.astype('float32'), 3)
    old_docs = [chunks_df_old.iloc[i]["doc_id"] for i in indices_old[0]]
    
    # Поиск в новой базе
    scores_new, indices_new = new_index.search(q_emb.astype('float32'), 3)
    new_docs = [chunks_df_new.iloc[i]["doc_id"] for i in indices_new[0]]
    
    return {
        "query": query,
        "before_retrieved": ", ".join(old_docs),
        "after_retrieved": ", ".join(new_docs),
        "changed": relevant_doc in new_docs and relevant_doc not in old_docs
    }

comparison = []
for q in new_queries:
    result = compare_retrieval(q["query"], q["relevant_doc"], index, updated_index, chunks_df, updated_chunks_df)
    comparison.append(result)

comparison_df = pd.DataFrame(comparison)
print("\n=== Сравнение до и после обновления ===\n")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
print("\nРезультаты сохранены в artifacts/retrieval_before_after_update.csv")

База знаний обновлена: 13 документов, 13 чанков

=== Сравнение до и после обновления ===

                            query       before_retrieved        after_retrieved  changed
   Как работает алгоритм K-Means? doc_03, doc_04, doc_10 doc_11, doc_03, doc_04     True
       Для чего используется PCA? doc_03, doc_02, doc_04 doc_12, doc_03, doc_13     True
Что такое функция активации ReLU? doc_02, doc_03, doc_08 doc_13, doc_02, doc_03     True

Результаты сохранены в artifacts/retrieval_before_after_update.csv


In [16]:
# Раздел 8: Mini-RAG (полный пайплайн)

def mini_rag(query: str, top_k: int = 3) -> Dict:
    """
    Mini-RAG пайплайн:
    1. Поиск релевантных чанков
    2. Формирование контекста
    3. Генерация ответа на основе контекста
    """
    # Шаг 1: Retrieval
    retrieved = search(query, top_k=top_k)
    
    # Шаг 2: Формирование контекста
    context_parts = []
    sources = []
    
    for _, row in retrieved.iterrows():
        context_parts.append(f"[{row['rank']}] {row['chunk_text']}")
        sources.append(f"{row['doc_id']} - {row['title']} (score: {row['score']:.3f})")
    
    context = "\n\n".join(context_parts)
    
    # Шаг 3: Генерация ответа (извлекаем наиболее релевантные предложения)
    # Для простоты используем extractive подход: берём топ-1 чанк и выделяем ключевое предложение
    
    if len(retrieved) > 0:
        # Берём лучший чанк
        best_chunk = retrieved.iloc[0]["chunk_text"]
        
        # Разбиваем на предложения
        sentences = re.split(r'[.!?]+', best_chunk)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
        
        if sentences:
            # Берём первое полное предложение как ответ
            answer = sentences[0] + "."
        else:
            answer = best_chunk[:200] + "..."
    else:
        answer = "Не удалось найти релевантную информацию в базе знаний."
    
    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "context": context,
        "retrieved_chunks": retrieved
    }

# Тестируем Mini-RAG
test_questions = [
    "В чем разница между L1 и L2 регуляризацией?",
    "Как Random Forest борется с переобучением?",
    "Что такое градиентный бустинг?"
]

print("\n=== Тестирование Mini-RAG ===\n")
for q in test_questions:
    result = mini_rag(q)
    print(f"Вопрос: {result['query']}")
    print(f"Ответ: {result['answer']}")
    print(f"Источники: {', '.join(result['sources'])}")
    print("-" * 60)


=== Тестирование Mini-RAG ===

Вопрос: В чем разница между L1 и L2 регуляризацией?
Ответ: Логистическая регрессия применяется для задач бинарной классификации.
Источники: doc_02 - Логистическая регрессия (score: 0.451), doc_03 - Деревья решений (score: 0.418), doc_09 - Регуляризация (score: 0.398)
------------------------------------------------------------
Вопрос: Как Random Forest борется с переобучением?
Ответ: Random Forest — это ансамбль деревьев решений, где каждое дерево обучается на случайном подвыборочном наборе данных (бэггинг) и случайном подмножестве признаков.
Источники: doc_05 - Ансамблевые методы: Random Forest (score: 0.814), doc_04 - Метод опорных векторов (SVM) (score: 0.397), doc_07 - Метрики классификации (score: 0.382)
------------------------------------------------------------
Вопрос: Что такое градиентный бустинг?
Ответ: Градиентный бустинг последовательно обучает слабые модели (обычно деревья), где каждая следующая модель исправляет ошибки предыдущей.
Источник

In [20]:
# Раздел 9: Сбор примеров работы Mini-RAG в CSV

rag_examples = []

# Берем 5 разнообразных вопросов для демонстрации
demo_questions = [
    "Что такое линейная регрессия?",
    "Объясните метод опорных векторов SVM",
    "Какие метрики используются для оценки классификации?",
    "Зачем нужна регуляризация в ML?",
    "Как работает градиентный бустинг?"
]

for q in demo_questions:
    result = mini_rag(q)
    rag_examples.append({
        "question": result["query"],
        "answer": result["answer"],
        "retrieved_sources": " | ".join(result["sources"])
    })

rag_examples_df = pd.DataFrame(rag_examples)
print("\n=== Примеры работы Mini-RAG ===\n")
print(rag_examples_df.to_string(index=False))

# Сохраняем в CSV
rag_examples_df.to_csv("artifacts/rag_examples.csv", index=False)
print("\n Примеры сохранены в artifacts/rag_examples.csv")


=== Примеры работы Mini-RAG ===

                                            question                                                                                                                                answer                                                                                                                                           retrieved_sources
                       Что такое линейная регрессия?                                                                 Логистическая регрессия применяется для задач бинарной классификации.                      doc_02 - Логистическая регрессия (score: 0.695) | doc_03 - Деревья решений (score: 0.629) | doc_01 - Линейная регрессия (score: 0.532)
                Объясните метод опорных векторов SVM                                                                     SVM строит гиперплоскость, максимизирующую отступ между классами.              doc_04 - Метод опорных векторов (SVM) (score: 0.625) | doc_07 - Метрики классифи

In [23]:
# Раздел 10: Анализ ошибок и ограничений

print("\n=== Анализ ошибок и ограничений ===\n")

# Находим запросы, где retrieval сработал плохо
weak_queries = []

for item in benchmark_queries:
    query = item["query"]
    relevant_id = item["relevant_doc_id"]
    
    # Проверяем hit@3
    results = search(query, top_k=3)
    retrieved_docs = results["doc_id"].tolist()
    
    if relevant_id not in retrieved_docs:
        weak_queries.append({
            "query": query[:80] + "..." if len(query) > 80 else query,
            "relevant_doc": relevant_id,
            "retrieved_docs": ", ".join(retrieved_docs),
            "problem": "Релевантный документ не попал в top-3"
        })
    elif results.iloc[0]["doc_id"] != relevant_id:
        weak_queries.append({
            "query": query[:80] + "..." if len(query) > 80 else query,
            "relevant_doc": relevant_id,
            "retrieved_docs": ", ".join(retrieved_docs),
            "problem": f"Релевантный документ на позиции {results[results['doc_id'] == relevant_id].iloc[0]['rank']}, не на первом месте"
        })

if weak_queries:
    weak_df = pd.DataFrame(weak_queries)
    print(" Найдены проблемные запросы:\n")
    print(weak_df.to_string(index=False))
else:
    print(" Все запросы успешно находят релевантный документ в top-3")

# Сохраняем анализ в отчёт
analysis_summary = {
    "total_queries": len(benchmark_queries),
    "failed_queries": len(weak_queries),
    "success_rate": f"{(len(benchmark_queries) - len(weak_queries)) / len(benchmark_queries) * 100:.1f}%",
    "main_limitations": [
        "Чувствительность к формулировке запроса",
        "Зависимость от параметров чанкинга",
        "Ограничения эмбеддинг-модели",
        "Примитивная генерация ответов",
        "Отсутствие reranking"
    ]
}

print("\n Краткая сводка анализа:")
print(f"   Всего запросов: {analysis_summary['total_queries']}")
print(f"   Проблемных запросов: {analysis_summary['failed_queries']}")
print(f"   Успешность hit@3: {analysis_summary['success_rate']}")


=== Анализ ошибок и ограничений ===

 Найдены проблемные запросы:

                                               query relevant_doc         retrieved_docs                                               problem
    Что такое линейная регрессия и как она работает?       doc_01 doc_02, doc_03, doc_01 Релевантный документ на позиции 3, не на первом месте
Какие метрики используются для оценки классификации?       doc_07 doc_02, doc_03, doc_04                 Релевантный документ не попал в top-3
                        Что означает R² в регрессии?       doc_08 doc_03, doc_08, doc_02 Релевантный документ на позиции 2, не на первом месте
         В чем разница между L1 и L2 регуляризацией?       doc_09 doc_02, doc_03, doc_09 Релевантный документ на позиции 3, не на первом месте
                Зачем нужно масштабировать признаки?       doc_10 doc_03, doc_10, doc_02 Релевантный документ на позиции 2, не на первом месте

 Краткая сводка анализа:
   Всего запросов: 10
   Проблемных запросов: 5
